# Plotting, smoothing and moving averages: lab notebook

**Statistical Modelling and Forecasting, Statistics background for forecasting**

Run the cells from top to bottom (Runtime > Run all). NumPy, pandas, SciPy, statsmodels and scikit-learn are already installed in Colab. Read each section, run it, and compare your output with the expected output at the end of the notebook.

In [ ]:
import warnings
warnings.filterwarnings("ignore")  # keeps the output tidy

In [ ]:
import numpy as np
import pandas as pd
import statsmodels.api as sm
import matplotlib.pyplot as plt

### 1. Load a monthly series (weekly CO2 averaged by month) and describe it

In [ ]:
# 1. Load a monthly series (weekly CO2 averaged by month) and describe it
co2 = sm.datasets.co2.load_pandas().data["co2"].resample("MS").mean().interpolate()
print(co2.describe().round(2).to_string())
print("average yearly growth (ppm per year):", round((co2["1998":"1998"].mean() - co2["1988":"1988"].mean()) / 10, 2))

### 2. Moving averages: trailing, centered, and the 2x12 average that removes the seasonal wave

In [ ]:
# 2. Moving averages: trailing, centered, and the 2x12 average that removes the seasonal wave
trail12 = co2.rolling(12).mean()
cent12 = co2.rolling(12, center=True).mean()
ma2x12 = cent12.rolling(2).mean().shift(-1)            # 2 x 12 centered average (even order)
print("values lost at the start / end for the 2x12 average:", int(ma2x12.isna().head(12).sum()), "/", int(ma2x12.isna().tail(12).sum()))
print("std of month-to-month changes: raw", round(co2.diff().std(), 3), "| 2x12 MA", round(ma2x12.diff().std(), 3))

### 3. The numerical problem by hand and with pandas

In [ ]:
# 3. The numerical problem by hand and with pandas
y = pd.Series([30, 33, 35, 34, 38, 41, 40, 44], index=range(1, 9), dtype=float)
print("centered 3-MA at t=5:", round(y.rolling(3, center=True).mean().loc[5], 3))
print("trailing 3-MA at t=8:", round(y.rolling(3).mean().loc[8], 3))
print("weighted MA (1,2,1)/4 at t=5:", (y.loc[4] + 2 * y.loc[5] + y.loc[6]) / 4)
t = np.arange(1, 9)
b, a = np.polyfit(t, y.values, 1)
print("linear trend: slope", round(b, 3), "intercept", round(a, 3), "| forecast t=9:", round(a + b * 9, 3))

### 4. Linear against quadratic trend on real GDP (quarterly)

In [ ]:
# 4. Linear against quadratic trend on real GDP (quarterly)
gdp = sm.datasets.macrodata.load_pandas().data["realgdp"].values
tt = np.arange(len(gdp))
for deg in (1, 2):
    coef = np.polyfit(tt, gdp, deg)
    resid = gdp - np.polyval(coef, tt)
    print(f"degree {deg}: residual std = {resid.std():.1f}, lag-1 autocorrelation of residuals = {np.corrcoef(resid[1:], resid[:-1])[0, 1]:.3f}")

### 5. Plot

In [ ]:
# 5. Plot
ax = co2.plot(alpha=0.5, label="monthly CO2")
ma2x12.plot(ax=ax, label="2x12 MA")
trail12.plot(ax=ax, label="trailing 12")
ax.legend(); plt.show()

## Expected output

Running all cells above should print:

```
count    526.00
mean     339.62
std       17.12
min      313.40
25%      324.02
50%      337.91
75%      354.54
max      373.80
average yearly growth (ppm per year): 1.52
values lost at the start / end for the 2x12 average: 6 / 6
std of month-to-month changes: raw 1.186 | 2x12 MA 0.053
centered 3-MA at t=5: 37.667
trailing 3-MA at t=8: 41.667
weighted MA (1,2,1)/4 at t=5: 37.75
linear trend: slope 1.845 intercept 28.571 | forecast t=9: 45.179
degree 1: residual std = 562.8, lag-1 autocorrelation of residuals = 0.994
degree 2: residual std = 217.7, lag-1 autocorrelation of residuals = 0.962
```

## Practice exercises

Try these in the cells below.

1. Use the seasonal series and set the window to 12, then to 6 and 5. When is the seasonal wave removed completely?
2. Compare centered and trailing MAs with k = 15. Measure the lag between the two curves and compare it with (k − 1)/2.
3. Choose the curved trend and compare the RMSE of the linear and quadratic fits against the true trend.

In [ ]:
# Exercise 1


In [ ]:
# Exercise 2


In [ ]:
# Exercise 3
